In [ ]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [ ]:
#ablation study 



In [1]:
# ================================================================
# GRAPH-GROUNDED WIND-TRANSPORT TEST v3: three-way comparison to
# separate the STATIC distance component from the DYNAMIC wind
# component, since the full edge-weight formula confounds them
# (distance decay is the same at every hour, so it can dominate
# "top target" selection regardless of wind).
#
# Three target-selection rules, same top-vs-control paired test:
#   (1) full formula:     decay(dist) * wind_alignment * wind_speed
#   (2) distance-only:    decay(dist) alone (nearest eligible neighbor,
#                          static per source, ignores wind entirely)
#   (3) wind-only:        wind_alignment * wind_speed alone (no decay
#                          term, so distance cannot drive the pick)
# All window-averaged over the 36h post-onset period, 2016-2019.
# ================================================================
import numpy as np
import pandas as pd
import json
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

RHO_KM = 250.0
HYBRID_THRESHOLD_KM = 20.0
region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
eligible_mask = (region_dist_km > HYBRID_THRESHOLD_KM)
region_decay = np.exp(-region_dist_km / RHO_KM)

EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
FORWARD_WINDOW = 36

pm25_station = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order]
dt_index_full = pm25_station.index

year_mask = (dt_index_full.year >= 2016) & (dt_index_full.year <= 2019)
dt_index = dt_index_full[year_mask]
n_time = len(dt_index)
pm25_station_arr = pm25_station.to_numpy()[year_mask]

wdir_station = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index_full).to_numpy().astype(float)[year_mask]
wspeed_station = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index_full).to_numpy().astype(float)[year_mask]
wdir_sin_station = np.sin(np.radians(wdir_station))
wdir_cos_station = np.cos(np.radians(wdir_station))

def regional_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float64)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_mean(pm25_station_arr)
region_windspeed = regional_mean(wspeed_station)
region_wdir_sin = regional_mean(wdir_sin_station)
region_wdir_cos = regional_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

def onset_indices(label_col):
    lab = np.nan_to_num(label_col, nan=0.0)
    prev = np.concatenate([[0], lab[:-1]])
    return np.where((lab == 1) & (prev == 0))[0]

region_onsets = [onset_indices(region_episode_label[:, r]) for r in range(n_regions)]
onset_bool = np.zeros((n_time, n_regions), dtype=bool)
for r in range(n_regions):
    onset_bool[region_onsets[r], r] = True
onset_cumsum = np.cumsum(onset_bool, axis=0)

def any_onset_in_window(region_idx, t_start_excl, t_end_incl):
    t_end_incl = min(t_end_incl, n_time - 1)
    if t_end_incl <= t_start_excl:
        return False
    hi = onset_cumsum[t_end_incl, region_idx]
    lo = onset_cumsum[t_start_excl, region_idx] if t_start_excl >= 0 else 0
    return bool(hi - lo > 0)

print(f"date range: {dt_index[0]} to {dt_index[-1]}  n_time={n_time}")
print(f"total onset events (all regions): {sum(len(o) for o in region_onsets)}")

# ================================================================
# For each A-onset event, compute window-averaged scores under all
# three ranking rules, run top-vs-control test for each.
# ================================================================
mode_records = {"full": [], "distance_only": [], "wind_only": []}
top_target_counts = {"full": {}, "distance_only": {}, "wind_only": {}}

for a in range(n_regions):
    dist_score_a = np.where(eligible_mask[a, :], region_decay[a, :], np.nan)
    dist_score_a[a] = np.nan

    for t_A in region_onsets[a]:
        w_end = min(t_A + FORWARD_WINDOW, n_time)
        if w_end <= t_A:
            continue
        wbt_a_window = region_wind_blows_toward[t_A:w_end, a]
        speed_a_window = region_windspeed[t_A:w_end, a]
        if np.all(np.isnan(wbt_a_window)) or np.all(np.isnan(speed_a_window)):
            continue

        bearing_row = region_bearing[a, :]
        cos_align_TR = np.maximum(np.cos(np.radians(wbt_a_window[:, None] - bearing_row[None, :])), 0.0)
        wind_only_TR = cos_align_TR * speed_a_window[:, None]           # no decay term
        full_TR = region_decay[a, :][None, :] * wind_only_TR             # full formula

        avg_full = np.nanmean(full_TR, axis=0)
        avg_wind_only = np.nanmean(wind_only_TR, axis=0)

        avg_full = np.where(eligible_mask[a, :], avg_full, np.nan); avg_full[a] = np.nan
        avg_wind_only = np.where(eligible_mask[a, :], avg_wind_only, np.nan); avg_wind_only[a] = np.nan

        if np.all(np.isnan(avg_full)):
            continue

        scores = {"full": avg_full, "distance_only": dist_score_a, "wind_only": avg_wind_only}

        for mode, score_arr in scores.items():
            if np.all(np.isnan(score_arr)):
                continue
            top_target = int(np.nanargmax(score_arr))
            top_target_counts[mode][region_names[a]] = top_target_counts[mode].get(region_names[a], {})
            top_target_counts[mode][region_names[a]][region_names[top_target]] = \
                top_target_counts[mode][region_names[a]].get(region_names[top_target], 0) + 1

            hit_top = any_onset_in_window(top_target, t_A, t_A + FORWARD_WINDOW)
            other_hits = [any_onset_in_window(r, t_A, t_A + FORWARD_WINDOW)
                          for r in range(n_regions) if r != a and r != top_target and not np.isnan(score_arr[r])]
            control_hit_rate = float(np.mean(other_hits)) if other_hits else np.nan
            mode_records[mode].append({"source": region_names[a], "target_top": region_names[top_target],
                                        "hit_top": bool(hit_top), "control_hit_rate": control_hit_rate})

# ================================================================
# Results for each mode
# ================================================================
print(f"\n{'='*25} THREE-WAY COMPARISON {'='*25}")
summary = {}
for mode in ["full", "distance_only", "wind_only"]:
    recs = mode_records[mode]
    hit_top_arr = np.array([1.0 if r["hit_top"] else 0.0 for r in recs])
    control_arr = np.array([r["control_hit_rate"] for r in recs])
    valid = ~np.isnan(control_arr)
    hit_top_arr, control_arr = hit_top_arr[valid], control_arr[valid]
    diff = hit_top_arr - control_arr

    t_stat, p_two = stats.ttest_1samp(diff, 0.0)
    p_one_sided = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    w_stat, p_wilcoxon = stats.wilcoxon(diff)

    print(f"\n--- {mode} ---")
    print(f"n_events={len(diff)}")
    print(f"top-target hit rate:  {hit_top_arr.mean():.4f}")
    print(f"control rate:         {control_arr.mean():.4f}")
    print(f"lift:                 {diff.mean():+.4f}")
    print(f"paired t-test: t={t_stat:.3f}  p_one_sided={p_one_sided:.6f}")
    print(f"Wilcoxon: p={p_wilcoxon:.6f}")

    summary[mode] = {"n_events": len(diff), "top_rate": float(hit_top_arr.mean()),
                      "control_rate": float(control_arr.mean()), "lift": float(diff.mean()),
                      "t_stat": float(t_stat), "p_one_sided": float(p_one_sided), "p_wilcoxon": float(p_wilcoxon)}

print(f"\n{'='*25} SUMMARY TABLE {'='*25}")
print(f"{'mode':16s} {'top_rate':>10s} {'control':>10s} {'lift':>8s} {'p_one_sided':>12s}")
for mode in ["full", "distance_only", "wind_only"]:
    s = summary[mode]
    print(f"{mode:16s} {s['top_rate']:10.4f} {s['control_rate']:10.4f} {s['lift']:+8.4f} {s['p_one_sided']:12.6f}")

with open(f"{BASE}/ablation_wind_vs_distance_decomposition.json", "w") as f:
    json.dump({"summary": summary, "top_target_counts": top_target_counts}, f, indent=2)
print(f"\nsaved to {BASE}/ablation_wind_vs_distance_decomposition.json")


date range: 2016-01-01 00:00:00 to 2019-12-31 23:00:00  n_time=35064
total onset events (all regions): 708

========================= THREE-WAY COMPARISON =========================

--- full ---
n_events=708
top-target hit rate:  0.3884
control rate:         0.2978
lift:                 +0.0906
paired t-test: t=5.582  p_one_sided=0.000000
Wilcoxon: p=0.000001

--- distance_only ---
n_events=708
top-target hit rate:  0.3686
control rate:         0.2992
lift:                 +0.0695
paired t-test: t=4.148  p_one_sided=0.000019
Wilcoxon: p=0.000584

--- wind_only ---
n_events=708
top-target hit rate:  0.3362
control rate:         0.3014
lift:                 +0.0347
paired t-test: t=2.224  p_one_sided=0.013229
Wilcoxon: p=0.181794

========================= SUMMARY TABLE =========================
mode               top_rate    control     lift  p_one_sided
full                 0.3884     0.2978  +0.0906     0.000000
distance_only        0.3686     0.2992  +0.0695     0.000019
wind_only   

In [2]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated:
# Jeju gets ZERO neighbor features (no nearest-region lookups), matching
# the graph models where Jeju has no cross-region edges. Uses the tuned
# config found to be optimal on both the 2018 and within-COVID splits
# (lr=0.05, max_depth=4, l2=0.0), reused rather than re-searched.
# Split: train=2016-2017, val=2018, test=2019.
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

# ---- nearest-3-region lookup, EXCLUDING Jeju entirely (no neighbors for Jeju,
# and Jeju is never used as a neighbor for anyone else either, matching the
# graph models where all of Jeju's edges -- in and out -- are zeroed) ----
region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

# ================================================================
# Build features: single-region (mean+last per channel + static) +
# nearest-3-region mean+last PM2.5 (ZERO for Jeju, and Jeju excluded
# as a candidate neighbor for everyone else)
# ================================================================
buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)  # mean+last PM25 for up to 3 neighbors -> 6 values
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        # (Jeju: neighbors=[] -> neighbor_feats stays all zero)
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

# ---- reused tuned config (optimal on both 2018 and within-COVID splits) ----
BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

print(f"\n{'='*20} FULL 4-MODEL COMPARISON (2019 test, Jeju isolated) {'='*20}")
print(f"wind_graph (Jeju isolated, 10-seed):  AUC-PR=0.7064 +/- 0.0085")
print(f"no_graph (unaffected, 10-seed):       AUC-PR=0.6805 +/- 0.0025")
print(f"minimal_gru (unaffected, 10-seed):    AUC-PR=0.6743 +/- 0.0063")
print(f"HistGB+3nn (Jeju isolated, 1 run):    AUC-PR={test_aucpr:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_2019.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_2019.json")


train hours: 17544  val hours: 8760  test hours: 8760
rows: train=297041 val=147713 test=147713  (positive rate train=0.0370)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated) ====================
test AUC-ROC=0.9340  AUC-PR=0.6109
precision=0.5274  recall=0.5769  MCC=0.5250

==================== FULL 4-MODEL COMPARISON (2019 test, Jeju isolated) ====================
wind_graph (Jeju isolated, 10-seed):  AUC-PR=0.7064 +/- 0.0085
no_graph (unaffected, 10-seed):       AUC-PR=0.6805 +/- 0.0025
minimal_gru (unaffected, 10-seed):    AUC-PR=0.6743 +/- 0.0063
HistGB+3nn (Jeju isolated, 1 run):    AUC-PR=0.6109

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_2019.json


In [3]:
# ================================================================
# JEJU-ISOLATION ABLATION, 2018 SPLIT: fair-tuned wind_graph pipeline
# (train=2016, val=2017, test=2018), wind_graph ONLY, with every
# edge (station AND region level) touching Jeju forced to zero weight.
# Uses the 2018-optimal tuned config (lr=0.001, dropout=0.5, weight_decay=0.0005).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2016, val=2017, test=2018 ----
split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016): {TRAIN_MASK.sum()}  val hours (2017): {(split_id_per_hour==1).sum()}  test hours (2018): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

# ---- ISOLATE JEJU: zero out every edge (station AND region level) that
# touches Jeju, in either direction ----
jeju_idx = region_names.index("Jeju")
jeju_station_mask = (station_region_idx == jeju_idx)
station_touches_jeju = jeju_station_mask[src_idx] | jeju_station_mask[dst_idx]
station_wind_edge_weight[:, station_touches_jeju] = 0.0
region_touches_jeju = (r_src_idx == jeju_idx) | (r_dst_idx == jeju_idx)
region_wind_edge_weight[:, region_touches_jeju] = 0.0
print(f"zeroed {station_touches_jeju.sum()}/{len(station_touches_jeju)} station edges and "
      f"{region_touches_jeju.sum()}/{len(region_touches_jeju)} region edges touching Jeju")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds, wind_graph ONLY, Jeju isolated, 2018-optimal config
# ================================================================
N_SEEDS = 10
results = {"wind_graph_jeju_isolated": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph_jeju_isolated", wind_ewt_s, wind_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r, lr=0.001, dropout=0.5, weight_decay=0.0005)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)

        per_region = {}
        for r in range(n_regions):
            y_r = ytraj2[:, r, :].max(axis=-1)
            score_r = test_probs[:, r, :].max(axis=-1)
            if y_r.sum() == 0:
                continue
            per_region[region_names[r]] = {
                "aucpr": float(average_precision_score(y_r, score_r)),
                "aucroc": float(roc_auc_score(y_r, score_r)),
            }

        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}  "
              f"Jeju_AUCPR={per_region.get('Jeju', {}).get('aucpr', float('nan')):.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
                                     "per_region": per_region})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph_jeju_isolated"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph_jeju_isolated"]])
jeju_aucpr = np.array([r["per_region"].get("Jeju", {}).get("aucpr", np.nan) for r in results["wind_graph_jeju_isolated"]])
jeju_aucroc = np.array([r["per_region"].get("Jeju", {}).get("aucroc", np.nan) for r in results["wind_graph_jeju_isolated"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS (Jeju isolated, 2018 split) {'='*20}")
print(f"overall  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"overall  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"Jeju     AUC-PR:  mean={jeju_aucpr.mean():.4f}  std={jeju_aucpr.std():.4f}  values={jeju_aucpr.round(4)}")
print(f"Jeju     AUC-ROC: mean={jeju_aucroc.mean():.4f}  std={jeju_aucroc.std():.4f}  values={jeju_aucroc.round(4)}")

print(f"\n{'='*20} COMPARE TO CONNECTED BASELINE (2018 split) {'='*20}")
print(f"overall wind_graph (connected, 10-seed mean):  AUC-PR=0.4628")
print(f"Jeju    wind_graph (ISOLATED, {N_SEEDS}-seed mean): AUC-PR={jeju_aucpr.mean():.4f}  AUC-ROC={jeju_aucroc.mean():.4f}")

print(f"\n{'='*20} PER-REGION AUC-PR, AVERAGED ACROSS {N_SEEDS} SEEDS {'='*20}")
for r in range(n_regions):
    name = region_names[r]
    vals = np.array([res["per_region"].get(name, {}).get("aucpr", np.nan) for res in results["wind_graph_jeju_isolated"]], dtype=float)
    print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")

with open(f"{BASE}/wind_graph_jeju_isolated_2018split_10seed.json", "w") as f:
    json.dump({"wind_graph_jeju_isolated": results["wind_graph_jeju_isolated"],
               "overall_aucpr_mean": float(wind_aucpr.mean()), "overall_aucpr_std": float(wind_aucpr.std()),
               "jeju_aucpr_mean": float(jeju_aucpr.mean()), "jeju_aucpr_std": float(jeju_aucpr.std())}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_jeju_isolated_2018split_10seed.json")


device=mps
train hours (2016): 8784  val hours (2017): 8760  test hours (2018): 8760
zeroed 140/21414 station edges and 32/272 region edges touching Jeju
windows: train=8713 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph_jeju_isolated ==========
  trained in 459s  val_AUCPR=0.4326  test_AUCROC=0.9287  test_AUCPR=0.4577  Jeju_AUCPR=0.1326

========== seed=1  model=wind_graph_jeju_isolated ==========
  trained in 452s  val_AUCPR=0.4557  test_AUCROC=0.9314  test_AUCPR=0.4793  Jeju_AUCPR=0.1536

========== seed=2  model=wind_graph_jeju_isolated ==========
  trained in 457s  val_AUCPR=0.4315  test_AUCROC=0.9292  test_AUCPR=0.4373  Jeju_AUCPR=0.1073

========== seed=3  model=wind_graph_jeju_isolated ==========
  trained in 459s  val_AUCPR=0.4425  test_AUCROC=0.9289  test_AUCPR=0.4680  Jeju_AUCPR=0.1683

========== seed=4  model=wind_graph_jeju_isolated ==========
  trained in 458s  val_AUCPR=0.4568  test_AUCROC=0.9319  test_AUCPR=0.4814  Jeju_

In [4]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated,
# 2018 split (train=2016, val=2017, test=2018). Reuses the tuned
# config (lr=0.05, max_depth=4, l2=0.0), same as found optimal on
# this exact split's single-region tree baseline and on within-COVID.
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()

# ---- SPLIT: train=2016, val=2017, test=2018 ----
split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated, 2018 split) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_2018.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_2018.json")


train hours: 8784  val hours: 8760  test hours: 8760
rows: train=148121 val=147713 test=147713  (positive rate train=0.0368)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated, 2018 split) ====================
test AUC-ROC=0.9110  AUC-PR=0.4013
precision=0.3831  recall=0.4414  MCC=0.3855

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_2018.json


In [ ]:
#2020-2021 split. 



In [1]:
# ================================================================
# JEJU-ISOLATION ABLATION, WITHIN-COVID-REGIME SPLIT:
# train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021. wind_graph ONLY,
# every edge (station AND region level) touching Jeju forced to zero
# weight. Uses the tuned config for this split
# (lr=0.001, dropout=0.5, weight_decay=0.0005).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2020): {TRAIN_MASK.sum()}  val hours (H1-2021): {(split_id_per_hour==1).sum()}  test hours (H2-2021): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

# ---- ISOLATE JEJU: zero out every edge (station AND region level) that
# touches Jeju, in either direction ----
jeju_idx = region_names.index("Jeju")
jeju_station_mask = (station_region_idx == jeju_idx)
station_touches_jeju = jeju_station_mask[src_idx] | jeju_station_mask[dst_idx]
station_wind_edge_weight[:, station_touches_jeju] = 0.0
region_touches_jeju = (r_src_idx == jeju_idx) | (r_dst_idx == jeju_idx)
region_wind_edge_weight[:, region_touches_jeju] = 0.0
print(f"zeroed {station_touches_jeju.sum()}/{len(station_touches_jeju)} station edges and "
      f"{region_touches_jeju.sum()}/{len(region_touches_jeju)} region edges touching Jeju")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds, wind_graph ONLY, Jeju isolated, within-COVID config
# ================================================================
N_SEEDS = 10
results = {"wind_graph_jeju_isolated": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph_jeju_isolated", wind_ewt_s, wind_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r, lr=0.001, dropout=0.5, weight_decay=0.0005)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)

        per_region = {}
        for r in range(n_regions):
            y_r = ytraj2[:, r, :].max(axis=-1)
            score_r = test_probs[:, r, :].max(axis=-1)
            if y_r.sum() == 0:
                continue
            per_region[region_names[r]] = {
                "aucpr": float(average_precision_score(y_r, score_r)),
                "aucroc": float(roc_auc_score(y_r, score_r)),
            }

        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}  "
              f"Jeju_AUCPR={per_region.get('Jeju', {}).get('aucpr', float('nan')):.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
                                     "per_region": per_region})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph_jeju_isolated"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph_jeju_isolated"]])
jeju_aucpr = np.array([r["per_region"].get("Jeju", {}).get("aucpr", np.nan) for r in results["wind_graph_jeju_isolated"]])
jeju_aucroc = np.array([r["per_region"].get("Jeju", {}).get("aucroc", np.nan) for r in results["wind_graph_jeju_isolated"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS (Jeju isolated, within-COVID split) {'='*20}")
print(f"overall  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"overall  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"Jeju     AUC-PR:  mean={jeju_aucpr.mean():.4f}  std={jeju_aucpr.std():.4f}  values={jeju_aucpr.round(4)}")
print(f"Jeju     AUC-ROC: mean={jeju_aucroc.mean():.4f}  std={jeju_aucroc.std():.4f}  values={jeju_aucroc.round(4)}")

print(f"\n{'='*20} COMPARE TO CONNECTED BASELINE (within-COVID split) {'='*20}")
print(f"overall wind_graph (connected, 10-seed mean):  AUC-PR=0.4904")
print(f"Jeju    wind_graph (ISOLATED, {N_SEEDS}-seed mean): AUC-PR={jeju_aucpr.mean():.4f}  AUC-ROC={jeju_aucroc.mean():.4f}")

print(f"\n{'='*20} PER-REGION AUC-PR, AVERAGED ACROSS {N_SEEDS} SEEDS {'='*20}")
for r in range(n_regions):
    name = region_names[r]
    vals = np.array([res["per_region"].get(name, {}).get("aucpr", np.nan) for res in results["wind_graph_jeju_isolated"]], dtype=float)
    print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")

with open(f"{BASE}/wind_graph_jeju_isolated_covidsplit_10seed.json", "w") as f:
    json.dump({"wind_graph_jeju_isolated": results["wind_graph_jeju_isolated"],
               "overall_aucpr_mean": float(wind_aucpr.mean()), "overall_aucpr_std": float(wind_aucpr.std()),
               "jeju_aucpr_mean": float(jeju_aucpr.mean()), "jeju_aucpr_std": float(jeju_aucpr.std())}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_jeju_isolated_covidsplit_10seed.json")


device=mps
train hours (2020): 8784  val hours (H1-2021): 4344  test hours (H2-2021): 4416
zeroed 140/21414 station edges and 32/272 region edges touching Jeju
windows: train=8713 val=4273 test=4345
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph_jeju_isolated ==========
  trained in 323s  val_AUCPR=0.3057  test_AUCROC=0.9676  test_AUCPR=0.5383  Jeju_AUCPR=nan

========== seed=1  model=wind_graph_jeju_isolated ==========
  trained in 315s  val_AUCPR=0.2805  test_AUCROC=0.9518  test_AUCPR=0.4272  Jeju_AUCPR=nan

========== seed=2  model=wind_graph_jeju_isolated ==========
  trained in 313s  val_AUCPR=0.3269  test_AUCROC=0.9497  test_AUCPR=0.4411  Jeju_AUCPR=nan

========== seed=3  model=wind_graph_jeju_isolated ==========
  trained in 314s  val_AUCPR=0.3218  test_AUCROC=0.9572  test_AUCPR=0.4847  Jeju_AUCPR=nan

========== seed=4  model=wind_graph_jeju_isolated ==========
  trained in 313s  val_AUCPR=0.3084  test_AUCROC=0.9510  test_AUCPR=0.5059  Jeju_AUCPR=

/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_2702/1661266546.py:515: RuntimeWarning: Mean of empty slice
  print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [1]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated,
# within-COVID-regime split (train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021).
# Reuses the tuned config (lr=0.05, max_depth=4, l2=0.0).
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated, within-COVID split) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_covid.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_covid.json")


train hours: 8784  val hours: 4344  test hours: 4416
rows: train=148121 val=72641 test=73865  (positive rate train=0.0134)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated, within-COVID split) ====================
test AUC-ROC=0.9602  AUC-PR=0.1584
precision=0.1740  recall=0.4631  MCC=0.2710

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_covid.json


In [1]:
#obtain key metrics at different forecast horizons. 

# ================================================================
# HORIZON-STRATIFIED EVALUATION (12h / 24h / 36h), 2019 test year.
# wind_graph: loaded from saved_wind_graph_2019/ (exact reproduction).
# no_graph, minimal_gru: retrained fresh (single seed=0, fair-tuned configs).
# Metrics per horizon: AUC-ROC, AUC-PR, precision, recall (at val
# F1-optimal threshold), and mean detection lead time (hours).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    """No graph convolution at all -- per-station Linear -> GRU."""
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges (needed for wind_graph inference and no_graph training) ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

# ---- normalization: use the SAVED wind_graph normalization for consistency ----
norm = np.load(f"{BASE}/saved_wind_graph_2019/normalization.npz")
t_mean, t_std = norm["t_mean"], norm["t_std"]
s_mean, s_std = norm["s_mean"], norm["s_std"]
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

# ================================================================
# MODEL 1: wind_graph -- load saved 2019 model, no retraining
# ================================================================
print(f"\n{'#'*15} loading saved wind_graph 2019 model {'#'*15}")
wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

wind_val_probs = predict_traj_probs_graph(wind_model, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
wind_test_probs = predict_traj_probs_graph(wind_model, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# MODEL 2: no_graph -- fresh training, fair-tuned config
# ================================================================
print(f"\n{'#'*15} training no_graph (seed=0, tuned config) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
lr, dropout, weight_decay = 0.001, 0.5, 0.0005
p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_graph(p1, zero_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch_graph(p1, zero_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
nograph_model = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
del p1; gc.collect()

opt2 = torch.optim.Adam(nograph_model.parameters(), lr=lr, weight_decay=weight_decay)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    run_p2_traj_epoch_graph(nograph_model, zero_ewt_s, zero_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
    run_p2_traj_epoch_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
    vp = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(nograph_model.state_dict())
nograph_model.load_state_dict(best_state)
nograph_model.eval()
print(f"no_graph val_aucpr={best_val_aucpr:.4f}")

nograph_val_probs = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, starts1_t)
nograph_test_probs = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X2_t, starts2_t)
del nograph_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# MODEL 3: minimal_gru -- fresh training, fair-tuned config
# ================================================================
print(f"\n{'#'*15} training minimal_gru (seed=0, tuned config) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
lr, dropout, weight_decay = 0.003, 0.3, 0.0005
p1 = StationGRU(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
    vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_lin)
minimal_model = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
del p1; gc.collect()

opt2 = torch.optim.Adam(minimal_model.parameters(), lr=lr, weight_decay=weight_decay)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    run_p2_traj_epoch_gru(minimal_model, X0_t, ytraj0_t, opt2, True)
    run_p2_traj_epoch_gru(minimal_model, X1_t, ytraj1_t, opt2, False)
    vp = predict_traj_probs_gru(minimal_model, X1_t)
    va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(minimal_model.state_dict())
minimal_model.load_state_dict(best_state)
minimal_model.eval()
print(f"minimal_gru val_aucpr={best_val_aucpr:.4f}")

minimal_val_probs = predict_traj_probs_gru(minimal_model, X1_t)
minimal_test_probs = predict_traj_probs_gru(minimal_model, X2_t)
del minimal_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# HORIZON-STRATIFIED METRICS for the 3 trajectory models
# ================================================================
def horizon_metrics(val_probs, val_labels, test_probs, test_labels, H, model_name):
    val_score_H = val_probs[..., :H].max(axis=-1).reshape(-1)
    val_label_H = val_labels[..., :H].max(axis=-1).reshape(-1)
    test_score_H = test_probs[..., :H].max(axis=-1).reshape(-1)
    test_label_H = test_labels[..., :H].max(axis=-1).reshape(-1)

    aucroc = roc_auc_score(test_label_H, test_score_H)
    aucpr = average_precision_score(test_label_H, test_score_H)

    fpr, tpr, thresholds = roc_curve(val_label_H, val_score_H)
    f1s = []
    for th in thresholds:
        pred = (val_score_H >= th).astype(int)
        p = precision_score(val_label_H, pred, zero_division=0)
        r = recall_score(val_label_H, pred, zero_division=0)
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    best_thresh = thresholds[np.argmax(f1s)]

    pred_test = (test_score_H >= best_thresh).astype(int)
    precision = precision_score(test_label_H, pred_test, zero_division=0)
    recall = recall_score(test_label_H, pred_test, zero_division=0)

    n_win, n_reg, _ = test_labels.shape
    lead_times = []
    n_true_onsets = 0
    for w in range(n_win):
        for r in range(n_reg):
            lab = test_labels[w, r, :H]
            if lab.max() == 0:
                continue
            n_true_onsets += 1
            onset_h = int(np.argmax(lab))
            prob = test_probs[w, r, :H]
            alarm_idx = np.where(prob >= best_thresh)[0]
            alarm_idx = alarm_idx[alarm_idx <= onset_h]
            if len(alarm_idx) > 0:
                alarm_h = alarm_idx[0]
                lead_times.append(onset_h - alarm_h)
    mean_lead = float(np.mean(lead_times)) if lead_times else float("nan")
    detect_rate = len(lead_times) / n_true_onsets if n_true_onsets else float("nan")

    print(f"  [{model_name}] H={H}h  AUC-ROC={aucroc:.4f}  AUC-PR={aucpr:.4f}  "
          f"precision={precision:.4f}  recall={recall:.4f}  "
          f"mean_lead={mean_lead:.2f}h  detect_rate={detect_rate:.2%}  n_onsets={n_true_onsets}")
    return {"H": H, "aucroc": float(aucroc), "aucpr": float(aucpr), "precision": float(precision),
            "recall": float(recall), "mean_lead_hours": mean_lead, "detect_rate": detect_rate,
            "n_true_onsets": n_true_onsets, "threshold": float(best_thresh)}

HORIZONS = [12, 24, 36]
results = {"wind_graph": [], "no_graph": [], "minimal_gru": []}

print(f"\n{'='*20} HORIZON-STRATIFIED METRICS {'='*20}")
for H in HORIZONS:
    results["wind_graph"].append(horizon_metrics(wind_val_probs, ytraj1, wind_test_probs, ytraj2, H, "wind_graph"))
for H in HORIZONS:
    results["no_graph"].append(horizon_metrics(nograph_val_probs, ytraj1, nograph_test_probs, ytraj2, H, "no_graph"))
for H in HORIZONS:
    results["minimal_gru"].append(horizon_metrics(minimal_val_probs, ytraj1, minimal_test_probs, ytraj2, H, "minimal_gru"))

with open(f"{BASE}/horizon_stratified_2019_graph_gru.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/horizon_stratified_2019_graph_gru.json")
print("Now run the separate tree-baseline script to complete the comparison.")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689

############### loading saved wind_graph 2019 model ###############

############### training no_graph (seed=0, tuned config) ###############
no_graph val_aucpr=0.4681

############### training minimal_gru (seed=0, tuned config) ###############
minimal_gru val_aucpr=0.4694

==================== HORIZON-STRATIFIED METRICS ====================
  [wind_graph] H=12h  AUC-ROC=0.9862  AUC-PR=0.8113  precision=0.7879  recall=0.6780  mean_lead=1.13h  detect_rate=67.63%  n_onsets=4745
  [wind_graph] H=24h  AUC-ROC=0.9741  AUC-PR=0.7686  precision=0.7114  recall=0.6811  mean_lead=3.19h  detect_rate=66.56%  n_onsets=6426
  [wind_graph] H=36h  AUC-ROC=0.9580  AUC-PR=0.7154  precision=0.7129  recall=0.5960  mean_lead=4.27h  detect_rate=58.01%  n_onsets=7929
  [no_graph] H=12h  AUC-ROC=0.9792  AUC-PR=0.7842  precision=0.7743  recall=0.6499  mean_lead=0.98h  detect_rate=

In [ ]:
#more seeds to remove variation. 


# ================================================================
# HORIZON-STRATIFIED EVALUATION (12h / 24h / 36h), 2019 test year,
# 5 SEEDS for wind_graph, no_graph, minimal_gru (all trained fresh
# with their respective tuned configs, for full methodological
# consistency across the three models -- no mixing loaded + trained).
# Reports mean +/- std across seeds per horizon per model.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def horizon_metrics(val_probs, val_labels, test_probs, test_labels, H):
    val_score_H = val_probs[..., :H].max(axis=-1).reshape(-1)
    val_label_H = val_labels[..., :H].max(axis=-1).reshape(-1)
    test_score_H = test_probs[..., :H].max(axis=-1).reshape(-1)
    test_label_H = test_labels[..., :H].max(axis=-1).reshape(-1)

    aucroc = roc_auc_score(test_label_H, test_score_H)
    aucpr = average_precision_score(test_label_H, test_score_H)

    fpr, tpr, thresholds = roc_curve(val_label_H, val_score_H)
    f1s = []
    for th in thresholds:
        pred = (val_score_H >= th).astype(int)
        p = precision_score(val_label_H, pred, zero_division=0)
        r = recall_score(val_label_H, pred, zero_division=0)
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    best_thresh = thresholds[np.argmax(f1s)]

    pred_test = (test_score_H >= best_thresh).astype(int)
    precision = precision_score(test_label_H, pred_test, zero_division=0)
    recall = recall_score(test_label_H, pred_test, zero_division=0)

    n_win, n_reg, _ = test_labels.shape
    lead_times = []
    n_true_onsets = 0
    for w in range(n_win):
        for r in range(n_reg):
            lab = test_labels[w, r, :H]
            if lab.max() == 0:
                continue
            n_true_onsets += 1
            onset_h = int(np.argmax(lab))
            prob = test_probs[w, r, :H]
            alarm_idx = np.where(prob >= best_thresh)[0]
            alarm_idx = alarm_idx[alarm_idx <= onset_h]
            if len(alarm_idx) > 0:
                lead_times.append(onset_h - alarm_idx[0])
    mean_lead = float(np.mean(lead_times)) if lead_times else float("nan")
    detect_rate = len(lead_times) / n_true_onsets if n_true_onsets else float("nan")

    return {"aucroc": float(aucroc), "aucpr": float(aucpr), "precision": float(precision),
            "recall": float(recall), "mean_lead_hours": mean_lead, "detect_rate": detect_rate}

HORIZONS = [12, 24, 36]
N_SEEDS = 5
CONFIGS = {
    "wind_graph": {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005, "ewt_s": wind_ewt_s, "ewt_r": wind_ewt_r},
    "no_graph": {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005, "ewt_s": zero_ewt_s, "ewt_r": zero_ewt_r},
}
MINIMAL_CFG = {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005}

all_results = {"wind_graph": {H: [] for H in HORIZONS}, "no_graph": {H: [] for H in HORIZONS}, "minimal_gru": {H: [] for H in HORIZONS}}

for seed in range(N_SEEDS):
    for model_name, cfg in CONFIGS.items():
        print(f"\n{'#'*10} seed={seed}  model={model_name} {'#'*10}")
        torch.manual_seed(seed); np.random.seed(seed)
        lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
        ewt_s, ewt_r = cfg["ewt_s"], cfg["ewt_r"]
        p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
        opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
        best_p1_val, best_p1_state = float("inf"), None
        for epoch in range(1, MAX_EPOCHS_P1 + 1):
            run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
            vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
            if vl < best_p1_val:
                best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
        p1.load_state_dict(best_p1_state)
        encoder_copy = copy.deepcopy(p1.station_conv)
        model = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
        del p1; gc.collect()

        opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
        best_val_aucpr, best_state = -1.0, None
        for epoch in range(1, MAX_EPOCHS_P2 + 1):
            run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
            run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
            vp = predict_traj_probs_graph(model, ewt_s, ewt_r, X1_t, starts1_t)
            va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
            if va > best_val_aucpr:
                best_val_aucpr, best_state = va, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        model.eval()
        print(f"  val_aucpr={best_val_aucpr:.4f}")

        val_probs = predict_traj_probs_graph(model, ewt_s, ewt_r, X1_t, starts1_t)
        test_probs = predict_traj_probs_graph(model, ewt_s, ewt_r, X2_t, starts2_t)
        del model; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        for H in HORIZONS:
            m = horizon_metrics(val_probs, ytraj1, test_probs, ytraj2, H)
            all_results[model_name][H].append(m)
            print(f"    H={H}h  AUC-PR={m['aucpr']:.4f}  AUC-ROC={m['aucroc']:.4f}  "
                  f"precision={m['precision']:.4f}  recall={m['recall']:.4f}  lead={m['mean_lead_hours']:.2f}h")

    # minimal_gru
    print(f"\n{'#'*10} seed={seed}  model=minimal_gru {'#'*10}")
    torch.manual_seed(seed); np.random.seed(seed)
    lr, dropout, weight_decay = MINIMAL_CFG["lr"], MINIMAL_CFG["dropout"], MINIMAL_CFG["weight_decay"]
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    model = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()

    opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch_gru(model, X0_t, ytraj0_t, opt2, True)
        run_p2_traj_epoch_gru(model, X1_t, ytraj1_t, opt2, False)
        vp = predict_traj_probs_gru(model, X1_t)
        va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    print(f"  val_aucpr={best_val_aucpr:.4f}")

    val_probs = predict_traj_probs_gru(model, X1_t)
    test_probs = predict_traj_probs_gru(model, X2_t)
    del model; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    for H in HORIZONS:
        m = horizon_metrics(val_probs, ytraj1, test_probs, ytraj2, H)
        all_results["minimal_gru"][H].append(m)
        print(f"    H={H}h  AUC-PR={m['aucpr']:.4f}  AUC-ROC={m['aucroc']:.4f}  "
              f"precision={m['precision']:.4f}  recall={m['recall']:.4f}  lead={m['mean_lead_hours']:.2f}h")

# ================================================================
# AGGREGATE: mean +/- std across 5 seeds, per model per horizon
# ================================================================
print(f"\n{'='*20} AGGREGATED ACROSS {N_SEEDS} SEEDS {'='*20}")
summary = {}
for model_name in ["wind_graph", "no_graph", "minimal_gru"]:
    summary[model_name] = {}
    for H in HORIZONS:
        recs = all_results[model_name][H]
        agg = {k: (float(np.mean([r[k] for r in recs])), float(np.std([r[k] for r in recs])))
               for k in ["aucroc", "aucpr", "precision", "recall", "mean_lead_hours", "detect_rate"]}
        summary[model_name][H] = agg
        print(f"[{model_name}] H={H}h  "
              f"AUC-PR={agg['aucpr'][0]:.4f}+/-{agg['aucpr'][1]:.4f}  "
              f"AUC-ROC={agg['aucroc'][0]:.4f}+/-{agg['aucroc'][1]:.4f}  "
              f"precision={agg['precision'][0]:.4f}+/-{agg['precision'][1]:.4f}  "
              f"recall={agg['recall'][0]:.4f}+/-{agg['recall'][1]:.4f}  "
              f"lead={agg['mean_lead_hours'][0]:.2f}+/-{agg['mean_lead_hours'][1]:.2f}h")

with open(f"{BASE}/horizon_stratified_2019_5seed.json", "w") as f:
    json.dump({"per_seed": all_results, "summary": summary}, f, indent=2)
print(f"\nsaved to {BASE}/horizon_stratified_2019_5seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689

########## seed=0  model=wind_graph ##########
  val_aucpr=0.5071
    H=12h  AUC-PR=0.8113  AUC-ROC=0.9860  precision=0.7942  recall=0.6643  lead=1.10h
    H=24h  AUC-PR=0.7693  AUC-ROC=0.9738  precision=0.7460  recall=0.6555  lead=3.01h
    H=36h  AUC-PR=0.7161  AUC-ROC=0.9575  precision=0.6889  recall=0.6215  lead=4.48h

########## seed=0  model=no_graph ##########
  val_aucpr=0.4681
    H=12h  AUC-PR=0.7842  AUC-ROC=0.9792  precision=0.7743  recall=0.6499  lead=0.98h
    H=24h  AUC-PR=0.7305  AUC-ROC=0.9639  precision=0.7422  recall=0.6078  lead=2.63h
    H=36h  AUC-PR=0.6777  AUC-ROC=0.9474  precision=0.6098  recall=0.6174  lead=4.56h

########## seed=0  model=minimal_gru ##########
  val_aucpr=0.4694
    H=12h  AUC-PR=0.7883  AUC-ROC=0.9809  precision=0.7488  recall=0.6752  lead=0.86h
    H=24h  AUC-PR=0.7374  AUC-ROC=0.9661  precision=0.7294  recall=